# Fine-Tuning Social-STGCNN — ECP Loss on zara2

**Goal:** Reduce collision rate on `zara2` while preserving ADE / FDE, by fine-tuning
the pre-trained baseline with a stronger ECP collision loss and larger `d_min`.

## Why `val_best.pth` alone is not enough

`val_best.pth` is saved based on **lowest NLL validation loss only** — no collision term.
This means it picks the epoch where the model best fits trajectory distribution, which is
typically the epoch where the collision-loss effect is **smallest** (the model has
partially recovered its NLL by learning to ignore the collision penalty).

To see the real collision-avoidance effect we therefore save and evaluate **two checkpoints** per run:

| Checkpoint | Saved when | What it captures |
|---|---|---|
| `val_best.pth` | Val NLL hits new minimum | Best trajectory accuracy — least collision benefit |
| `epoch_final.pth` | Last training epoch | Maximum collision-loss effect — some ADE/FDE cost |

Comparing the two reveals the true ADE vs CollisionRate tradeoff for each (d_min, lambda) config.

## Experiment grid

- **Loss:** ECP (acts on full predicted distribution, not just means)
- **d_min ∈ {0.2, 0.5, 1.0} m** — 0.2 m is the original threshold (almost never fires);
  0.5 m matches the real p1 pedestrian spacing (~100× more activations);
  1.0 m activates on ~10% of training pairs
- **lambda_col ∈ {5, 20}** — original lambda=1 was dominated by NLL; 5 is moderate, 20 is aggressive
- All 6 runs start from `baseline/social-stgcnn-zara2/val_best.pth`

| | lambda = 5 | lambda = 20 |
|---|---|---|
| d_min = 0.2 m | run 1 | run 2 |
| d_min = 0.5 m | run 3 | run 4 |
| d_min = 1.0 m | run 5 | run 6 |

**Before running:** Runtime → Change runtime type → **GPU** (T4 / L4 / A100).

## 1. Environment setup

In [ ]:
!nvidia-smi
import torch
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available(),
      '-', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no GPU')

In [ ]:
!pip install -q 'networkx>=3.0' 'tqdm>=4.60' pandas

## 2. Mount Google Drive & set project directory

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/DLproject-Social-STGCNN'
assert os.path.isdir(PROJECT_DIR), f'Project not found at {PROJECT_DIR}'
os.chdir(PROJECT_DIR)
print('CWD:', os.getcwd())
!ls

## 3. Imports

In [ ]:
import os, copy, pickle, math, glob
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

import numpy as np
import torch
import torch.optim as optim
import torch.distributions.multivariate_normal as torchdist
from torch.utils.data import DataLoader
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

from model import social_stgcnn
from utils import TrajectoryDataset, seq_to_nodes, nodes_rel_to_nodes_abs
from metrics import ade, fde, collision_metrics
from collision_losses import collision_loss

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

## 4. Experiment configuration

In [ ]:
# ── Fixed settings ──────────────────────────────────────────────────────
DATASET         = 'zara2'
LOSS_FN         = 'ecp'
ECP_K           = 10
BASELINE_CKPT   = './baseline/social-stgcnn-zara2/val_best.pth'
BASELINE_ARGS   = './baseline/social-stgcnn-zara2/args.pkl'
FINETUNE_ROOT   = './finetune'
FINETUNE_EPOCHS = 100
FINETUNE_LR     = 0.001
BATCH_SIZE      = 128
KSTEPS          = 20
D_COL_EVAL      = 0.2   # evaluation collision radius — keep consistent with all prior results

# ── Sweep grid ───────────────────────────────────────────────────────────
D_MIN_LIST   = [0.2, 0.5, 1.0]
LAMBDA_LIST  = [5, 20]

CONFIGS = [
    {'d_min': d, 'lambda_col': l}
    for d in D_MIN_LIST
    for l in LAMBDA_LIST
]

def run_tag(cfg):
    return f'social-stgcnn-{DATASET}-{LOSS_FN}-dmin{cfg["d_min"]}-lam{cfg["lambda_col"]}'

def run_dir(cfg):
    return os.path.join(FINETUNE_ROOT, run_tag(cfg))

print(f'Fine-tuning {len(CONFIGS)} configs on {DATASET} with {LOSS_FN} loss:')
for c in CONFIGS:
    print(f'  d_min={c["d_min"]}  lambda_col={c["lambda_col"]}  ->  {run_dir(c)}')

## 5. Load shared data loaders

Also verifies how many training pairs fall within each `d_min` threshold — this confirms
the collision loss will actually fire during training.

In [ ]:
data_root = f'./datasets/{DATASET}/'

with open(BASELINE_ARGS, 'rb') as f:
    base_args = pickle.load(f)

obs_len  = base_args.obs_seq_len   # 8
pred_len = base_args.pred_seq_len  # 12

print('Building datasets ...', flush=True)
dset_train = TrajectoryDataset(data_root+'train/', obs_len=obs_len, pred_len=pred_len,
                               skip=1, norm_lap_matr=True)
dset_val   = TrajectoryDataset(data_root+'val/',   obs_len=obs_len, pred_len=pred_len,
                               skip=1, norm_lap_matr=True)
dset_test  = TrajectoryDataset(data_root+'test/',  obs_len=obs_len, pred_len=pred_len,
                               skip=1, norm_lap_matr=True)

loader_train = DataLoader(dset_train, batch_size=1, shuffle=True,  num_workers=0)
loader_val   = DataLoader(dset_val,   batch_size=1, shuffle=False, num_workers=0)
loader_test  = DataLoader(dset_test,  batch_size=1, shuffle=False, num_workers=0)

print(f'Train: {len(dset_train)}  Val: {len(dset_val)}  Test: {len(dset_test)} scenes')

# How many training pairs fall inside each d_min? — confirms the loss will fire
print('\nPairwise distance coverage in training data:')
all_dists = []
for path in glob.glob(data_root + 'train/*.txt'):
    raw = np.loadtxt(path)
    for fr in np.unique(raw[:, 0]):
        xy = raw[raw[:, 0] == fr][:, 2:]
        N = len(xy)
        for i in range(N):
            for j in range(i+1, N):
                all_dists.append(np.linalg.norm(xy[i] - xy[j]))
all_dists = np.array(all_dists)
print(f'  Total ped pairs : {len(all_dists):,}')
for thr in [0.2, 0.5, 1.0]:
    n = (all_dists < thr).sum()
    print(f'  Pairs < {thr} m  : {n:5d}  ({100*n/len(all_dists):.3f}%)')

## 6. Model builder

In [ ]:
def build_model_from_baseline():
    m = social_stgcnn(
        n_stgcnn    = base_args.n_stgcnn,
        n_txpcnn    = base_args.n_txpcnn,
        output_feat = base_args.output_size,
        seq_len     = base_args.obs_seq_len,
        kernel_size = base_args.kernel_size,
        pred_seq_len= base_args.pred_seq_len,
    ).to(device)
    m.load_state_dict(torch.load(BASELINE_CKPT, map_location=device))
    return m

_m = build_model_from_baseline()
print(f'Model parameters: {sum(p.numel() for p in _m.parameters()):,}')
del _m

## 7. Fine-tuning function

**Two checkpoints are saved per run:**
- `val_best.pth`    — epoch with lowest val NLL (best trajectory accuracy)
- `epoch_final.pth` — weights after the very last epoch (maximum collision-loss effect)

Validation always uses NLL only — consistent with the baseline — so `val_best.pth` picks
the most accurate model, not the most collision-aware one.  `epoch_final.pth` gives the
other end of the tradeoff.

In [ ]:
def bivariate_loss(V_pred, V_trgt):
    normx = V_trgt[:,:,0] - V_pred[:,:,0]
    normy = V_trgt[:,:,1] - V_pred[:,:,1]
    sx    = torch.exp(V_pred[:,:,2])
    sy    = torch.exp(V_pred[:,:,3])
    corr  = torch.tanh(V_pred[:,:,4])
    sxsy  = sx * sy
    z     = (normx/sx)**2 + (normy/sy)**2 - 2*corr*normx*normy/sxsy
    negRho = 1 - corr**2
    result = torch.exp(-z / (2*negRho))
    denom  = 2 * math.pi * sxsy * torch.sqrt(negRho)
    return -torch.log(torch.clamp(result/denom, min=1e-20)).mean()


def fine_tune(cfg, verbose=True):
    d_min      = cfg['d_min']
    lambda_col = cfg['lambda_col']
    tag        = run_tag(cfg)
    ckpt_dir   = run_dir(cfg)
    os.makedirs(ckpt_dir, exist_ok=True)

    model     = build_model_from_baseline()
    optimizer = optim.SGD(model.parameters(), lr=FINETUNE_LR)

    history = {'train_loss': [], 'train_nll': [], 'train_col': [], 'val_loss': []}
    best    = {'epoch': -1, 'val_loss': float('inf')}

    # Persist args so evaluate_model() can reload this checkpoint
    saved_args = copy.deepcopy(base_args)
    saved_args.collision_loss = LOSS_FN
    saved_args.lambda_col     = lambda_col
    saved_args.d_min          = d_min
    saved_args.ecp_k          = ECP_K
    with open(os.path.join(ckpt_dir, 'args.pkl'), 'wb') as f:
        pickle.dump(saved_args, f)

    loader_len = len(loader_train)
    turn_point = int(loader_len/BATCH_SIZE)*BATCH_SIZE + loader_len%BATCH_SIZE - 1

    for epoch in range(FINETUNE_EPOCHS):

        # ── Train ─────────────────────────────────────────────────────
        model.train()
        loss_sum = nll_sum = col_sum = 0.0
        n_updates = batch_count = 0
        is_fst = True

        for cnt, batch in enumerate(loader_train):
            batch_count += 1
            batch = [t.to(device) for t in batch]
            obs_traj, _, _, _, _, _, V_obs, A_obs, V_tr, _ = batch

            optimizer.zero_grad()
            V_pred, _ = model(V_obs.permute(0,3,1,2), A_obs.squeeze())
            V_pred = V_pred.permute(0,2,3,1).squeeze()
            V_tr   = V_tr.squeeze()

            start_pos = obs_traj[0, :, :, -1]   # (N, 2)
            l_nll = bivariate_loss(V_pred, V_tr)
            l_col = collision_loss(LOSS_FN, V_pred, start_pos, d_min=d_min, K=ECP_K)
            l     = l_nll + lambda_col * l_col

            if batch_count % BATCH_SIZE != 0 and cnt != turn_point:
                if is_fst: loss_acc = l; nll_acc = l_nll; col_acc = l_col; is_fst = False
                else:       loss_acc += l; nll_acc += l_nll; col_acc += l_col
            else:
                (loss_acc / BATCH_SIZE).backward()
                optimizer.step()
                loss_sum += (loss_acc / BATCH_SIZE).item()
                nll_sum  += (nll_acc  / BATCH_SIZE).item()
                col_sum  += (col_acc  / BATCH_SIZE).item()
                n_updates += 1
                is_fst = True

        n_updates = max(n_updates, 1)
        history['train_loss'].append(loss_sum / n_updates)
        history['train_nll'].append(nll_sum  / n_updates)
        history['train_col'].append(col_sum  / n_updates)

        # ── Validate (NLL only) ────────────────────────────────────────
        model.eval()
        vl_sum = vl_acc = 0.0
        v_updates = v_count = 0
        is_fst = True
        vloader_len = len(loader_val)
        v_turn = int(vloader_len/BATCH_SIZE)*BATCH_SIZE + vloader_len%BATCH_SIZE - 1

        with torch.no_grad():
            for cnt, batch in enumerate(loader_val):
                v_count += 1
                batch = [t.to(device) for t in batch]
                _, _, _, _, _, _, V_obs, A_obs, V_tr, _ = batch
                V_pred, _ = model(V_obs.permute(0,3,1,2), A_obs.squeeze())
                V_pred = V_pred.permute(0,2,3,1).squeeze()
                l = bivariate_loss(V_pred, V_tr.squeeze())
                if v_count % BATCH_SIZE != 0 and cnt != v_turn:
                    if is_fst: vl_acc = l; is_fst = False
                    else:       vl_acc += l
                else:
                    vl_sum += (vl_acc / BATCH_SIZE).item()
                    v_updates += 1; is_fst = True

        avg_val = vl_sum / max(v_updates, 1)
        history['val_loss'].append(avg_val)

        # ── Checkpoint: best NLL ───────────────────────────────────────
        if avg_val < best['val_loss']:
            best['val_loss'] = avg_val
            best['epoch']    = epoch
            torch.save(model.state_dict(), os.path.join(ckpt_dir, 'val_best.pth'))

        # ── Checkpoint: final epoch (always overwrite) ─────────────────
        torch.save(model.state_dict(), os.path.join(ckpt_dir, 'epoch_final.pth'))

        if verbose and (epoch % 10 == 0 or epoch == FINETUNE_EPOCHS - 1):
            print(f'  [{tag}] ep {epoch:3d}  '
                  f'nll={history["train_nll"][-1]:.5f}  '
                  f'col={history["train_col"][-1]:.5f}  '
                  f'val={avg_val:.5f}  '
                  f'best_nll_ep={best["epoch"]}')

    with open(os.path.join(ckpt_dir, 'metrics.pkl'),          'wb') as f: pickle.dump(history, f)
    with open(os.path.join(ckpt_dir, 'constant_metrics.pkl'), 'wb') as f: pickle.dump(best, f)

    print(f'  Saved val_best (ep {best["epoch"]}) + epoch_final -> {ckpt_dir}')
    return history

print('fine_tune() defined.')

## 8. Evaluation helper

In [ ]:
def evaluate_model(model_path, K=KSTEPS, d_col=D_COL_EVAL):
    """Evaluate a single checkpoint on the zara2 test set."""
    ckpt_dir = os.path.dirname(model_path)
    with open(os.path.join(ckpt_dir, 'args.pkl'), 'rb') as f:
        args = pickle.load(f)

    m = social_stgcnn(
        n_stgcnn=args.n_stgcnn, n_txpcnn=args.n_txpcnn,
        output_feat=args.output_size, seq_len=args.obs_seq_len,
        kernel_size=args.kernel_size, pred_seq_len=args.pred_seq_len,
    ).to(device)
    m.load_state_dict(torch.load(model_path, map_location=device))
    m.eval()

    ade_ls, fde_ls, col_avg_ls, col_minade_ls = [], [], [], []

    with torch.no_grad():
        for batch in loader_test:
            batch = [t.to(device) for t in batch]
            obs_traj, _, obs_traj_rel, _, _, _, V_obs, A_obs, V_tr, _ = batch
            num_objs = obs_traj_rel.shape[1]

            V_pred, _ = m(V_obs.permute(0,3,1,2), A_obs.squeeze())
            V_pred = V_pred.permute(0,2,3,1).squeeze()[:, :num_objs, :]
            V_tr   = V_tr.squeeze()[:, :num_objs, :]

            sx   = torch.exp(V_pred[:,:,2])
            sy   = torch.exp(V_pred[:,:,3])
            corr = torch.tanh(V_pred[:,:,4])
            cov  = torch.zeros(*V_pred.shape[:2], 2, 2, device=device)
            cov[:,:,0,0] = sx*sx; cov[:,:,0,1] = corr*sx*sy
            cov[:,:,1,0] = corr*sx*sy; cov[:,:,1,1] = sy*sy
            mvn = torchdist.MultivariateNormal(V_pred[:,:,:2], cov)

            V_x     = seq_to_nodes(obs_traj.cpu().numpy().copy())
            V_x_abs = nodes_rel_to_nodes_abs(
                V_obs.cpu().numpy().squeeze().copy(), V_x[0].copy())
            V_y_abs = nodes_rel_to_nodes_abs(
                V_tr.cpu().numpy().squeeze().copy(), V_x[-1].copy())

            ade_pp = {n: [] for n in range(num_objs)}
            fde_pp = {n: [] for n in range(num_objs)}
            samples_scene = []

            for _ in range(K):
                samp_abs = nodes_rel_to_nodes_abs(
                    mvn.sample().cpu().numpy().squeeze().copy(), V_x[-1].copy())
                samples_scene.append(samp_abs[:, :num_objs, :])
                for n in range(num_objs):
                    ade_pp[n].append(ade([samp_abs[:,n:n+1,:]], [V_y_abs[:,n:n+1,:]], [1]))
                    fde_pp[n].append(fde([samp_abs[:,n:n+1,:]], [V_y_abs[:,n:n+1,:]], [1]))

            for n in range(num_objs):
                ade_ls.append(min(ade_pp[n]))
                fde_ls.append(min(fde_pp[n]))

            col = collision_metrics(samples_scene, V_y_abs[:,:num_objs,:], d_col=d_col)
            col_avg_ls.append(col['ColRate_avg'])
            col_minade_ls.append(col['ColRate_minADE'])

    return {
        'ADE':            sum(ade_ls)        / len(ade_ls),
        'FDE':            sum(fde_ls)        / len(fde_ls),
        'ColRate_avg':    sum(col_avg_ls)    / len(col_avg_ls),
        'ColRate_minADE': sum(col_minade_ls) / len(col_minade_ls),
    }

print('evaluate_model() defined.')

## 9. Run fine-tuning

6 configs × ~10-15 min each on T4.  Estimated total: **~1–1.5 hours**.

In [ ]:
all_histories = {}

for i, cfg in enumerate(CONFIGS):
    sep = '='*60
    print(f'\n{sep}')
    print(f'Config {i+1}/{len(CONFIGS)}: d_min={cfg["d_min"]}  lambda_col={cfg["lambda_col"]}')
    print(sep)
    hist = fine_tune(cfg, verbose=True)
    all_histories[run_tag(cfg)] = hist

print('\nAll fine-tuning runs complete.')

## 10. Training curves

In [ ]:
n_cols = 3
n_rows = math.ceil(len(CONFIGS) / n_cols)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 3.5*n_rows), squeeze=False)

for idx, cfg in enumerate(CONFIGS):
    hist = all_histories[run_tag(cfg)]
    ax   = axes[idx // n_cols][idx % n_cols]
    ep   = range(len(hist['train_loss']))
    ax.plot(ep, hist['train_nll'], label='Train NLL',  color='steelblue')
    ax.plot(ep, hist['train_col'], label='Train ECP',  color='darkorange', linestyle='--')
    ax.plot(ep, hist['val_loss'],  label='Val NLL',    color='crimson',    linestyle=':')
    ax.set_title(f'd_min={cfg["d_min"]}  lam={cfg["lambda_col"]}', fontsize=10)
    ax.set_xlabel('Epoch'); ax.legend(fontsize=7); ax.grid(True, alpha=0.3)

for j in range(len(CONFIGS), n_rows * n_cols):
    axes[j // n_cols][j % n_cols].set_visible(False)

fig.suptitle('Fine-tuning Curves (zara2, ECP loss)', fontsize=13)
plt.tight_layout()
plt.savefig('finetune_training_curves.png', dpi=120, bbox_inches='tight')
plt.show()

## 11. Evaluate all checkpoints

For each config we evaluate **two** checkpoints:
- `val_best.pth`    — best NLL epoch (most accurate, least collision benefit)
- `epoch_final.pth` — last training epoch (most collision benefit, some ADE cost)

In [ ]:
results = []

# ── Baseline ────────────────────────────────────────────────────────────
print('Evaluating baseline ...')
r = evaluate_model(BASELINE_CKPT)
r.update({'Model': 'Baseline', 'Checkpoint': 'val_best',
          'd_min': None, 'lambda_col': None})
results.append(r)
print(f'  ADE={r["ADE"]:.4f}  FDE={r["FDE"]:.4f}  '
      f'ColRate_avg={r["ColRate_avg"]:.4f}  ColRate_minADE={r["ColRate_minADE"]:.4f}')

# ── Fine-tuned: both checkpoints ────────────────────────────────────────
for cfg in CONFIGS:
    for ckpt_file, ckpt_label in [('val_best.pth',    'best NLL'),
                                   ('epoch_final.pth', 'final ep')]:
        path = os.path.join(run_dir(cfg), ckpt_file)
        lbl  = f'ECP d={cfg["d_min"]} lam={cfg["lambda_col"]} [{ckpt_label}]'
        print(f'Evaluating {lbl} ...')
        r = evaluate_model(path)
        r.update({'Model': lbl, 'Checkpoint': ckpt_label,
                  'd_min': cfg['d_min'], 'lambda_col': cfg['lambda_col']})
        results.append(r)
        print(f'  ADE={r["ADE"]:.4f}  FDE={r["FDE"]:.4f}  '
              f'ColRate_avg={r["ColRate_avg"]:.4f}  ColRate_minADE={r["ColRate_minADE"]:.4f}')

df = pd.DataFrame(results).set_index('Model')
for col in ['ADE','FDE','ColRate_avg','ColRate_minADE']:
    df[col] = pd.to_numeric(df[col])

base_ade = df.loc['Baseline', 'ADE']
base_fde = df.loc['Baseline', 'FDE']
base_col = df.loc['Baseline', 'ColRate_avg']
df['ΔADE']     = df['ADE']          - base_ade
df['ΔFDE']     = df['FDE']          - base_fde
df['ΔColRate']  = df['ColRate_avg'] - base_col

print('\nDone.')

## 12. Results table

In [ ]:
display_cols = ['Checkpoint','ADE','FDE','ColRate_avg','ColRate_minADE','ΔADE','ΔFDE','ΔColRate']
fmt = {'ADE':'{:.4f}','FDE':'{:.4f}',
       'ColRate_avg':'{:.4f}','ColRate_minADE':'{:.4f}',
       'ΔADE':'{:+.4f}','ΔFDE':'{:+.4f}','ΔColRate':'{:+.4f}'}

styled = (df[display_cols]
    .style
    .format(fmt)
    .background_gradient(subset=['ColRate_avg'], cmap='RdYlGn_r')
    .background_gradient(subset=['ADE'],         cmap='RdYlGn_r')
    .set_caption(
        'zara2 test results — val_best vs epoch_final for each config.\n'
        'val_best = best NLL checkpoint; epoch_final = last training epoch.')
)
display(styled)

## 13. Tradeoff plot: Collision Rate vs ADE / FDE

- `val_best` checkpoints: **filled** markers
- `epoch_final` checkpoints: **open** markers (same colour/shape)

The arrow from filled to open shows the collision-avoidance gain at the cost of accuracy.
The ideal region is **bottom-left** of the baseline star.

In [ ]:
# colour = d_min, marker = lambda_col — must match CONFIGS exactly
col_map    = {0.2: '#e74c3c', 0.5: '#f39c12', 1.0: '#3498db'}
marker_map = {5: 'o', 20: 's'}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, y_col, y_lbl in [(axes[0], 'ADE', 'ADE (m)'), (axes[1], 'FDE', 'FDE (m)')]:

    bx = df.loc['Baseline', 'ColRate_avg']
    by = df.loc['Baseline', y_col]
    ax.axvline(bx, color='gray', linestyle='--', linewidth=1, alpha=0.6)
    ax.axhline(by, color='gray', linestyle='--', linewidth=1, alpha=0.6)
    ax.scatter([bx], [by], marker='*', s=250, color='black', zorder=6, label='Baseline')
    ax.annotate('Baseline', (bx, by), xytext=(6, 4), textcoords='offset points', fontsize=8)

    for cfg in CONFIGS:
        c  = col_map[cfg['d_min']]
        mk = marker_map[cfg['lambda_col']]

        lbl_best  = f'ECP d={cfg["d_min"]} lam={cfg["lambda_col"]} [best NLL]'
        lbl_final = f'ECP d={cfg["d_min"]} lam={cfg["lambda_col"]} [final ep]'

        rx_best  = df.loc[lbl_best,  'ColRate_avg'];  ry_best  = df.loc[lbl_best,  y_col]
        rx_final = df.loc[lbl_final, 'ColRate_avg'];  ry_final = df.loc[lbl_final, y_col]

        # Arrow: val_best (filled) -> epoch_final (open)
        ax.annotate('', xy=(rx_final, ry_final), xytext=(rx_best, ry_best),
                    arrowprops=dict(arrowstyle='->', color=c, lw=1.2, alpha=0.6))
        ax.scatter([rx_best],  [ry_best],  color=c, marker=mk, s=90, zorder=5)
        ax.scatter([rx_final], [ry_final], facecolors='none', edgecolors=c,
                   marker=mk, s=90, linewidths=1.8, zorder=5)
        ax.annotate(f'd={cfg["d_min"]} l={cfg["lambda_col"]}',
                    (rx_final, ry_final), xytext=(5, 3),
                    textcoords='offset points', fontsize=6.5, color=c)

    ax.set_xlabel('ColRate_avg  (lower = better)', fontsize=10)
    ax.set_ylabel(f'{y_lbl}  (lower = better)', fontsize=10)
    ax.set_title(f'Collision Rate vs {y_lbl}  (zara2)', fontsize=11)
    ax.grid(True, alpha=0.3)

    legend_els = (
        [Line2D([0],[0], marker='o', color='w', markerfacecolor=v,
                markersize=8, label=f'd_min={k} m') for k, v in col_map.items()] +
        [Line2D([0],[0], marker=v, color='gray',
                markersize=8, label=f'lambda={k}') for k, v in marker_map.items()] +
        [Line2D([0],[0], marker='o', color='gray', markersize=7,
                label='filled = val_best'),
         Line2D([0],[0], marker='o', color='gray', markersize=7,
                fillstyle='none', label='open = epoch_final')]
    )
    ax.legend(handles=legend_els, fontsize=7.5, loc='upper right')

plt.tight_layout()
plt.savefig('finetune_tradeoff.png', dpi=130, bbox_inches='tight')
plt.show()
print('Saved finetune_tradeoff.png')

## 14. Summary: best config

Find the config + checkpoint that reduces collision rate the most while keeping ΔADE < +0.05 m.

In [ ]:
ADE_BUDGET = 0.05   # max allowed ADE increase over baseline

candidates = df[(df['ΔADE'] <= ADE_BUDGET) & (df.index != 'Baseline')].copy()

if candidates.empty:
    print(f'No config reduces collision rate within ΔADE <= {ADE_BUDGET} m.')
    print('Consider relaxing ADE_BUDGET or reducing FINETUNE_EPOCHS.')
else:
    best_cfg = candidates['ΔColRate'].idxmin()
    row = candidates.loc[best_cfg]
    print(f'Best config within ΔADE <= {ADE_BUDGET} m:')
    print(f'  Model       : {best_cfg}')
    print(f'  ADE         : {row["ADE"]:.4f}  (ΔADE={row["ΔADE"]:+.4f})')
    print(f'  FDE         : {row["FDE"]:.4f}  (ΔFDE={row["ΔFDE"]:+.4f})')
    print(f'  ColRate_avg : {row["ColRate_avg"]:.4f}  (ΔColRate={row["ΔColRate"]:+.4f})')
    print(f'  ColRate_minADE: {row["ColRate_minADE"]:.4f}')

## 15. Save results & persist to Drive

In [ ]:
df.to_csv('finetune_zara2_ecp_results.csv')
print('Saved finetune_zara2_ecp_results.csv')
print(df[['Checkpoint','ADE','FDE','ColRate_avg','ΔADE','ΔFDE','ΔColRate']].to_string())

In [ ]:
# Only needed if code lives in /content/ (not Drive mount)
# !cp -r ./finetune /content/drive/MyDrive/DLproject-Social-STGCNN/
# !cp finetune_zara2_ecp_results.csv finetune_tradeoff.png finetune_training_curves.png \
#     /content/drive/MyDrive/DLproject-Social-STGCNN/
print('Done.')